## Cell 1 — test API key


In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("ANTHROPIC_API_KEY")

if api_key:
    print("API key loaded successfully!")
    print(f"Key starts with: {api_key[:10]}...")
else:
    print("API key not found — check your .env file")

API key loaded successfully!
Key starts with: sk-ant-api...


# Cell 2 — test Claude API connection

In [2]:
# Cell 2 — test Claude API connection
import anthropic

client = anthropic.Anthropic(api_key=api_key)

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=100,
    messages=[
        {"role": "user", "content": "Say hello in one sentence."}
    ]
)

print(response.content[0].text)

Hello! It's nice to meet you.


## Cell 3 — load layer 2 data


In [7]:
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os

load_dotenv()

# load the daily sentiment data from layer 2
# run this from the same folder as your layer2 notebook
import sys
sys.path.append('.')

# rebuild the daily data from layer 2
# paste your layer 2 daily dataframe here or load from CSV
# easiest approach — save from layer 2 first

daily = pd.read_csv("daily_sentiment.csv", parse_dates=["published"])
print(daily.head())
print(f"\nTotal days: {len(daily)}")
print(f"High volume days: {daily['high_volume'].sum()}")

   published  avg_sentiment  article_count  positive  negative  neutral  \
0 2026-03-02      -0.325061             60         8        30       22   
1 2026-03-03      -0.377513             55         6        30       19   
2 2026-03-04      -0.314382             46         8        23       15   
3 2026-03-05      -0.260268             66        12        31       23   
4 2026-03-06      -0.416032             85        12        49       24   

   heat_score  high_volume  
0    0.283783        False  
1    0.088320        False  
2   -0.263513        False  
3    0.518338        False  
4    1.261096        False  

Total days: 27
High volume days: 2


## Cell 4 — detect anomalies


In [35]:
def detect_anomalies(daily):
    
    # calculate sentiment baseline
    mean_sentiment = daily["avg_sentiment"].mean()
    std_sentiment = daily["avg_sentiment"].std()
    
    # flag days where sentiment dropped sharply OR volume spiked
    anomalies = daily[
        (daily["high_volume"] == True) | ## high volumn is when the articular published 1.5sd greater than the avg
        (daily["avg_sentiment"] < mean_sentiment - 1.5 * std_sentiment)
    ].copy()

    # label which threshold was breached
    anomalies['sentiment_breach_threshold'] = anomalies['avg_sentiment'] < (mean_sentiment-1.5*std_sentiment)
    anomalies['both_metrics_breached'] = (anomalies['sentiment_breach_threshold'] == True) & (anomalies['high_volume'] == True)
    anomalies = anomalies.sort_values("avg_sentiment")
    
    print(f"Baseline sentiment: {mean_sentiment:.3f}")
    print(f"Anomaly threshold: {mean_sentiment - 1.5 * std_sentiment:.3f}")
    print(f"\nDays flagged for investigation: {len(anomalies)}")
    print()
    
    for _, row in anomalies.iterrows():
        print(f"{row['published'].strftime('%b %d')} "
              f"| sentiment: {row['avg_sentiment']:.3f} "
              f"| sentiment threshold: {row['avg_sentiment']:.3f} "
              f"| articles: {row['article_count']} "
              f"| high volume: {row['high_volume']} "
              f"| sentiment breached: {row['sentiment_breach_threshold']}"
              f" both_metrics_breached: {row['both_metrics_breached']}")
        
    return anomalies

anomalies = detect_anomalies(daily)

Baseline sentiment: -0.335
Anomaly threshold: -0.522

Days flagged for investigation: 3

Mar 22 | sentiment: -0.723 | sentiment threshold: -0.723 | articles: 11 | high volume: False | sentiment breached: True both_metrics_breached: False
Mar 19 | sentiment: -0.379 | sentiment threshold: -0.379 | articles: 95 | high volume: True | sentiment breached: False both_metrics_breached: False
Mar 18 | sentiment: -0.216 | sentiment threshold: -0.216 | articles: 96 | high volume: True | sentiment breached: False both_metrics_breached: False


## Cell 5 — Claude investigation agent


In [36]:
client = anthropic.Anthropic(api_key=api_key)

def investigate_anomaly(row):
    date_str = row["published"].strftime("%B %d, %Y")
    sentiment = row["avg_sentiment"]
    articles = row["article_count"]
    high_vol = row["high_volume"]
    
    prompt = f"""You are a profession financial news senior analyst investigating a sentiment anomaly.

Date: {date_str}
Average sentiment score: {sentiment:.3f} (scale: -1.0 very negative to +1.0 very positive)
Article count: {articles}
Unusually high volume: {high_vol}

Based on your knowledge of financial markets, political news, world news, and Federal Reserve news, investigate what likely caused this sentiment pattern on this date. Consider:
- Fed announcements or speeches
- Inflation data releases
- Market reactions
- Economic indicators released that day
- major world news that gets everyone's attention 

Provide a concise 3-4 sentence investigation report explaining what likely happened."""

    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=300,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    
    return response.content[0].text

# test on the most negative day first
most_negative = anomalies.iloc[0]
print(f"Investigating {most_negative['published'].strftime('%B %d, %Y')}...")
print()
report = investigate_anomaly(most_negative)
print(report)

Investigating March 22, 2026...

**INVESTIGATION REPORT - March 22, 2026 Sentiment Anomaly**

The significantly negative sentiment score of -0.723 on March 22, 2026 likely stems from a hawkish Federal Reserve policy announcement or Chair Powell speech indicating unexpected rate hikes to combat persistent inflation. This date falls near the typical March FOMC meeting timeframe, and such dovish-to-hawkish pivots historically trigger sharp negative market reactions as investors reprice growth expectations and discount rates. The moderate article volume (11 pieces) suggests the news was impactful but not extraordinarily surprising, pointing to Fed policy guidance that disappointed markets expecting more accommodative monetary policy. Alternative catalysts could include disappointing inflation data or concerning geopolitical developments, but Fed-related news remains the most probable driver given the date and sentiment intensity.


## # Cell 6 — agent with web search

In [48]:
import time
import anthropic

def investigate_with_search(row):
    date_str = row["published"].strftime("%B %d, %Y")
    sentiment = row["avg_sentiment"]
    articles = row["article_count"]

    messages = [{
        "role": "user",
        "content": f"""You are a senior financial analyst. Search the web for Federal Reserve and inflation news on {date_str}.

Context:
- Date: {date_str}
- Sentiment score: {sentiment:.3f} (-1.0 = very negative, +1.0 = very positive)
- Articles published: {articles}

After searching, respond in exactly this format:

KEYWORDS:
3-5 keywords capturing the main themes

SUMMARY:
1-2 sentences explaining what happened and why sentiment hit {sentiment:.3f}

INVESTMENT WATCHOUT:
1-2 sentences on what investors should consider in terms of the best course of action. Be specific about asset classes and sectors.

"""
    }]

    max_turns = 6
    turns = 0
    final_report = ""

    while turns < max_turns:
        try:
            response = client.messages.create(
                model="claude-sonnet-4-20250514",
                max_tokens=700,
                tools=[{
                    "type": "web_search_20250305",
                    "name": "web_search"
                }],
                messages=messages
            )

            turns += 1

            for block in response.content:
                if hasattr(block, "text") and block.text:
                    final_report = block.text

            messages.append({
                "role": "assistant",
                "content": response.content
            })

            if response.stop_reason == "end_turn" and len(final_report) > 200:
                return final_report

            if response.stop_reason == "end_turn" and len(final_report) <= 200:
                messages.append({
                    "role": "user",
                    "content": "Please complete your full investigation report."
                })

            time.sleep(5)

        except anthropic.RateLimitError:
            print("Rate limit hit — waiting 60 seconds...")
            time.sleep(60)
            continue

    return final_report if final_report else "Investigation incomplete"

# run all anomaly days
print("Running agentic investigation...\n")
print("=" * 60)

for _, row in anomalies.iterrows():
    date_str = row["published"].strftime("%B %d, %Y")
    print(f"\nInvestigating {date_str}...")
    print(f"Sentiment: {row['avg_sentiment']:.3f} | Articles: {row['article_count']}")
    print("-" * 40)
    report = investigate_with_search(row)
    print(report)
    print("=" * 60)
    print("Waiting 60 seconds...")
    time.sleep(60)

Running agentic investigation...


Investigating March 22, 2026...
Sentiment: -0.723 | Articles: 11
----------------------------------------
With oil potentially reaching $170-200 per barrel if the Strait stays closed, creating a stagflationary shock that could shift central bank policies and election outcomes, defensive positioning in healthcare, consumer staples, and inflation-protected securities (TIPS) becomes critical.
Waiting 60 seconds...

Investigating March 19, 2026...
Sentiment: -0.379 | Articles: 95
----------------------------------------
Rate limit hit — waiting 60 seconds...
.

---

## CONCLUSION

The March 19, 2026 developments represent a critical inflection point where geopolitical tensions, energy supply disruptions, and monetary policy constraints converge to create significant downside risks for financial markets. The -0.379 sentiment score accurately reflects investor concerns about the sustainability of current equity valuations in an environment of diminishing Fe

In [49]:

print(os.path.exists("daily_sentiment.csv"))

True
